# ocean: 1D

1D ocean time series data

**coordinate(s)**:
* tavg-u-hm-sea
    - tavg: time average
    - u: ocean surface
    - hm: horizontal mean/sum
    - sea: ocean domain

In [ ]:
## Import libraries
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl
import sys
import os
import glob
from IPython.display import HTML, display

sys.path.append(os.getcwd())
from utils import read_variables, read_compound_names

In [ ]:
# parameters for the cmorized data
cmorout=''                  # root for cmorized data, e.g., '/scratch/$USER/cmorout'
source_id      = ''         # model name, e.g., 'NorESM3-LM'
experiment_id  = ''         # experiment name, e.g., 'historical', 'ssp585', 'piControl'
variant_label  = ''         # variant label, e.g., 'r1i1p1f1'
grid_label     = ''         # grid label, e.g., 'gn', 'gr', 'g999'
version        = ''         # version, e.g., 'v20260601'

In [ ]:
# data path
data_path = os.path.join(cmorout, source_id, experiment_id, version)

# load methods for plotting and set defaults
methods =read_variables('data/methods.txt')
#print(methods.keys())

---
**List of datasets:** \
(datasets which are not presented/cmorized have no link.)

In [ ]:
# load compound names

coords = ('tavg-u-hm-sea')
cnames = read_compound_names('data/variables.nml')
# examples of compound names:
# cnames = ['ocean.sos.tavg-u-hm-sea.mon.glb', 'ocean.masscello.tavg-u-hm-sea.mon.glb']
for cname in cnames:
    realm = cname.split('.')[0]
    var = cname.split('.')[1]
    coord = cname.split('.')[2]
    freq = cname.split('.')[3]
    
    if realm != 'ocean' or coord not in coords:
        continue
    else:
        data_file = var+'_'+coord+'_*_'+grid_label+'_'+source_id+'_'+experiment_id+'_'+variant_label+'_*.nc'
        if not glob.glob(os.path.join(data_path, data_file)):
            print(cname)
            continue
        else:
            display(HTML(f'<a href="#{cname}">{cname}</a>'))


---
**Datasets validated:**

In [ ]:
# loop through compound names and plot
for cname in cnames:
    if cname not in methods.keys():
        print(f"{cname} not found in methods.txt, using default methods for plotting.")
    else:
        if methods[cname] is not None:
            if 'vertical' in methods[cname].keys():
                mth_vert = methods[cname]['vertical']

            if 'timeseries' in methods[cname].keys():
                mth_ts = methods[cname]['timeseries']

            if 'cmap' in methods[cname].keys():
                mth_cmap = methods[cname]['cmap']

    realm = cname.split('.')[0]
    var = cname.split('.')[1]
    coord = cname.split('.')[2]
    freq = cname.split('.')[3]

    if realm != 'ocean' or coord not in coords:
        continue

    data_file = var+'_'+coord+'_*_'+grid_label+'_'+source_id+'_'+experiment_id+'_'+variant_label+'_*.nc'

    if not glob.glob(os.path.join(data_path, data_file)):
        continue

    #with xr.open_mfdataset(os.path.join(data_path, data_file)) as ds:
    data_file = glob.glob(os.path.join(data_path, data_file))[0]
    with xr.open_dataset(os.path.join(data_path, data_file)) as ds:
        if var in ds:
            data = ds[var]
        else:
            continue

    data1d = data
    display(HTML(f'<div id="{cname}"></div>'))
    print(f'\033[1m{cname}\033[0m')
    print(f'long name: {data1d.long_name} ({data1d.units})')
    print(f'NorESM->CMOR: {data1d.attrs["original_name"]} -> {var}')
    if 'history' in data1d.attrs:
        print(f'history: {data1d.attrs["history"]}')
    if 'comment' in data1d.attrs:
        print(f'comment: {data1d.attrs["comment"]}')

    fig = plt.figure(figsize=(16, 3), dpi=96)
    ax2 = fig.add_subplot(111)
    if data1d.sizes["time"] < 2:
        ax2.text(0.1, 0.8, (f"global {mth_ts} value: {data1d.values[0]} {data1d.units}"))
        ax2.text(0.1, 0.6, ("less than 2 time steps."))
        ax2.text(0.1, 0.4, ("skipping timeseries plot."))
    else:
        data1d.plot(ax=ax2)
    plt.suptitle(data1d.long_name)

    plt.tight_layout()
    plt.show()

    del data, data1d
    del ax2, fig